In [1]:
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [2]:
%pip install --upgrade certifi

Note: you may need to restart the kernel to use updated packages.


In [3]:
import os
import certifi

os.environ["SSL_CERT_FILE"] = certifi.where()

In [4]:
import pandas as pd

url = "https://raw.githubusercontent.com/mricardo89/data-mining/main/Unit-II/datasets/googleplaystore.csv"
df = pd.read_csv(url)

df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,"10,000+",Free,0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,"500,000+",Free,0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,"5,000,000+",Free,0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,"50,000,000+",Free,0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,"100,000+",Free,0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [5]:
df.shape

(10841, 13)

## Pregunta A - aplicaciones duplicadas

In [6]:
df.duplicated(subset=['App']).sum()

np.int64(1181)

In [7]:
df.drop_duplicates(subset=['App'], keep='first', inplace=True)

In [8]:
df.duplicated(subset=['App']).sum()

np.int64(0)

#### Respuesta (A)

Se encontraron 1,181 registros duplicados tomando como referencia el nombre de la aplicación. Se eliminaron usando `drop_duplicates()` y se conservó únicamente la primera aparición de cada app.

Después de realizar la limpieza ya no quedaron nombres duplicados en el dataset.

## Pregunta B - convertir installs a num

In [9]:
df['Installs'].head(10)

0        10,000+
1       500,000+
2     5,000,000+
3    50,000,000+
4       100,000+
5        50,000+
6        50,000+
7     1,000,000+
8     1,000,000+
9        10,000+
Name: Installs, dtype: object

#### Respuesta (B) 1

La columna `Installs` contiene valores almacenados como texto debido a caracteres como `+` y `,`. Para poder utilizar estos datos en operaciones matemáticas es necesario eliminar estos caracteres y convertir la columna a un tipo numérico.

In [10]:
df['Installs'] = (
    df['Installs']
    .str.replace('+', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(int)
)

ValueError: invalid literal for int() with base 10: 'Free'

In [11]:
df[df['Installs'] == 'Free']

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
10472,Life Made WI-Fi Touchscreen Photo Frame,1.9,19.0,3.0M,"1,000+",Free,0,Everyone,NaN,"February 11, 2018",1.0.19,4.0 and up,NaN


#### Respuesta (B) 2
Al intentar convertir la columna `Installs` a entero se encontró el valor `"Free"`, el cual no corresponde al formato esperado de esta columna.

Al revisar el registro se encontró que una fila del dataset tiene sus datos desfasados, por lo que se considera un registro incorrecto y se elimina antes de continuar con la conversión.

In [12]:
df = df[df['Installs'] != 'Free'].copy()

In [13]:
df[df['Installs'] == 'Free']

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver


In [14]:
df['Installs'] = (
    df['Installs']
    .str.replace('+', '', regex=False)
    .str.replace(',', '', regex=False)
    .astype(int)
)

In [15]:
df['Installs'].head()

0       10000
1      500000
2     5000000
3    50000000
4      100000
Name: Installs, dtype: int64

In [16]:
df['Installs'].mean()

np.float64(7777506.732270421)

#### Respuesta (B) 3
Después de eliminar los caracteres `+` y `,`, se intentó convertir la columna `Installs` a tipo entero. Durante la conversión se encontró un registro con el valor `"Free"`, lo cual indicaba que esa fila estaba mal estructurada.

Se eliminó el registro incorrecto y posteriormente se realizó nuevamente la conversión de la columna.

El promedio obtenido fue de aproximadamente 7.78 millones de instalaciones por aplicación. Este resultado es una aproximación, ya que valores como `1,000,000+` se convierten en `1,000,000`.

## Pregunta C - Limpiar `Price`

In [17]:
df['Price'].unique()

array(['0', '$4.99', '$3.99', '$6.99', '$1.49', '$2.99', '$7.99', '$5.99',
       '$3.49', '$1.99', '$9.99', '$7.49', '$0.99', '$9.00', '$5.49',
       '$10.00', '$24.99', '$11.99', '$79.99', '$16.99', '$14.99',
       '$1.00', '$29.99', '$12.99', '$2.49', '$10.99', '$1.50', '$19.99',
       '$15.99', '$33.99', '$74.99', '$39.99', '$3.95', '$4.49', '$1.70',
       '$8.99', '$2.00', '$3.88', '$25.99', '$399.99', '$17.99',
       '$400.00', '$3.02', '$1.76', '$4.84', '$4.77', '$1.61', '$2.50',
       '$1.59', '$6.49', '$1.29', '$5.00', '$13.99', '$299.99', '$379.99',
       '$37.99', '$18.99', '$389.99', '$19.90', '$8.49', '$1.75',
       '$14.00', '$4.85', '$46.99', '$109.99', '$154.99', '$3.08',
       '$2.59', '$4.80', '$1.96', '$19.40', '$3.90', '$4.59', '$15.46',
       '$3.04', '$4.29', '$2.60', '$3.28', '$4.60', '$28.99', '$2.95',
       '$2.90', '$1.97', '$200.00', '$89.99', '$2.56', '$30.99', '$3.61',
       '$394.99', '$1.26', '$1.20', '$1.04'], dtype=object)

In [18]:
df['Price'] = (
    df['Price']
    .str.replace('$', '', regex=False)
    .astype(float)
)

In [19]:
df['Price'].head()

0    0.0
1    0.0
2    0.0
3    0.0
4    0.0
Name: Price, dtype: float64

#### Respuesta (C)

La columna `Price` contenía el símbolo `$`, por lo que sus valores no podían utilizarse directamente como números. 
Se eliminó el símbolo de dólar mediante `str.replace()` y posteriormente se convirtió la columna a tipo `float`, permitiendo realizar operaciones matemáticas con los precios.

## Pregunta D - outliers de precio

In [20]:
df['Price'].max()

400.0

In [21]:
df[df['Price'] > 200]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
4197,most expensive app (H),FAMILY,4.3,6,1.5M,100,Paid,399.99,Everyone,Entertainment,"July 16, 2018",1.0,7.0 and up
4362,💎 I'm rich,LIFESTYLE,3.8,718,26M,10000,Paid,399.99,Everyone,Lifestyle,"March 11, 2018",1.0.0,4.4 and up
4367,I'm Rich - Trump Edition,LIFESTYLE,3.6,275,7.3M,10000,Paid,400.00,Everyone,Lifestyle,"May 3, 2018",1.0.1,4.1 and up
5351,I am rich,LIFESTYLE,3.8,3547,1.8M,100000,Paid,399.99,Everyone,Lifestyle,"January 12, 2018",2.0,4.0.3 and up
5354,I am Rich Plus,FAMILY,4.0,856,8.7M,10000,Paid,399.99,Everyone,Entertainment,"May 19, 2018",3.0,4.4 and up
5355,I am rich VIP,LIFESTYLE,3.8,411,2.6M,10000,Paid,299.99,Everyone,Lifestyle,"July 21, 2018",1.1.1,4.3 and up
5356,I Am Rich Premium,FINANCE,4.1,1867,4.7M,50000,Paid,399.99,Everyone,Finance,"November 12, 2017",1.6,4.0 and up
5357,I am extremely Rich,LIFESTYLE,2.9,41,2.9M,1000,Paid,379.99,Everyone,Lifestyle,"July 1, 2018",1.0,4.0 and up
5358,I am Rich!,FINANCE,3.8,93,22M,1000,Paid,399.99,Everyone,Finance,"December 11, 2017",1.0,4.1 and up
5359,I am rich(premium),FINANCE,3.5,472,965k,5000,Paid,399.99,Everyone,Finance,"May 1, 2017",3.4,4.4 and up


#### Respuesta (D) 1
El precio máximo encontrado en el dataset fue de 400. 
Al filtrar las aplicaciones con precios superiores a 200 se encontraron principalmente aplicaciones como "I am Rich", cuyo propósito y precio no representan un modelo de negocio normal.

Estos valores pueden considerarse outliers y podrían afectar negativamente el entrenamiento de un modelo de Machine Learning, haciendo que el modelo interprete estos precios extremos como comportamientos normales.

In [22]:
df = df[df['Price'] < 50].copy()

In [23]:
df['Price'].max()

46.99

In [24]:
df[df['Price'] >= 50]

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver


#### Respuesta (D) 2
Se eliminaron del dataset las aplicaciones con precios iguales o superiores a 50 para evitar que precios extremos influyan en el futuro modelo predictivo. 
De esta manera, el dataset conserva únicamente aplicaciones con precios considerados más representativos del mercado.

In [25]:
df.to_csv('playstore_limpio.csv', index=False)

In [26]:
print("Dataset guardado correctamente.")

Dataset guardado correctamente.


In [27]:
df.head()

,App,Category,Rating,Reviews,Size,Installs,Type,Price,Content Rating,Genres,Last Updated,Current Ver,Android Ver
0,Photo Editor & Candy Camera & Grid & ScrapBook,ART_AND_DESIGN,4.1,159,19M,10000,Free,0.0,Everyone,Art & Design,"January 7, 2018",1.0.0,4.0.3 and up
1,Coloring book moana,ART_AND_DESIGN,3.9,967,14M,500000,Free,0.0,Everyone,Art & Design;Pretend Play,"January 15, 2018",2.0.0,4.0.3 and up
2,"U Launcher Lite – FREE Live Cool Themes, Hide ...",ART_AND_DESIGN,4.7,87510,8.7M,5000000,Free,0.0,Everyone,Art & Design,"August 1, 2018",1.2.4,4.0.3 and up
3,Sketch - Draw & Paint,ART_AND_DESIGN,4.5,215644,25M,50000000,Free,0.0,Teen,Art & Design,"June 8, 2018",Varies with device,4.2 and up
4,Pixel Draw - Number Art Coloring Book,ART_AND_DESIGN,4.3,967,2.8M,100000,Free,0.0,Everyone,Art & Design;Creativity,"June 20, 2018",1.1,4.4 and up


In [28]:
df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 9636 entries, 0 to 10840
Data columns (total 13 columns):
 #   Column          Non-Null Count  Dtype  
---  ------          --------------  -----  
 0   App             9636 non-null   object 
 1   Category        9636 non-null   object 
 2   Rating          8180 non-null   float64
 3   Reviews         9636 non-null   object 
 4   Size            9636 non-null   object 
 5   Installs        9636 non-null   int64  
 6   Type            9635 non-null   object 
 7   Price           9636 non-null   float64
 8   Content Rating  9636 non-null   object 
 9   Genres          9636 non-null   object 
 10  Last Updated    9636 non-null   object 
 11  Current Ver     9628 non-null   object 
 12  Android Ver     9634 non-null   object 
dtypes: float64(2), int64(1), object(10)
memory usage: 1.0+ MB
